In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import cobra

# Add parent directory to Python path
sys.path.append(os.path.abspath('..'))

# First, import the base kinGEMs package
import kinGEMs

# Import necessary functions from kinGEMs package
from kinGEMs.dataset import load_model, convert_to_irreversible, prepare_model_data, retrieve_sequences, map_metabolites, merge_substrate_sequences, process_merged_data_with_folds
from kinGEMs.modeling.optimize import run_optimization_with_dataframe
from kinGEMs.plots import plot_flux_distribution  # Assuming this function exists

# Define paths
data_dir = "../data"
raw_data_dir = os.path.join(data_dir, "raw")
model_path = os.path.join(raw_data_dir, "e_coli_core.xml")
interim_data_dir = os.path.join(data_dir, "interim")
processed_data_dir = os.path.join(data_dir, "processed")
CPIPred_data_dir = os.path.join(interim_data_dir, "CPI-Pred predictions")

# # Step 1: Load the model
# print("Loading model...")
# model = load_model(model_path)
# print(f"Model loaded with {len(model.reactions)} reactions and {len(model.metabolites)} metabolites")

# # Step 2: Convert to irreversible model
# print("Converting to irreversible model...")
# irrev_model = convert_to_irreversible(model)
# print(f"Irreversible model created with {len(irrev_model.reactions)} reactions")

# Step 1: Prepare model data (metabolite and sequence mapping)
print("Preparing model data...")
# Define output paths
substrates_output = os.path.join(interim_data_dir, "ecoli_substrates.csv")
sequences_output = os.path.join(interim_data_dir, "ecoli_sequences.csv")

# Prepare model data
irrev_model, substrate_df, sequences_df = prepare_model_data(
    model_path=model_path,
    substrates_output=substrates_output,
    sequences_output=sequences_output,
    organism='E coli'
)

display(substrate_df)
display(sequences_df)

# Step 2: Merge substrate and sequence data
print("Merging substrate and sequence data...")
merged_data_output = os.path.join(interim_data_dir, "ecoli_merged_data.csv")
merged_data = merge_substrate_sequences(
    substrate_df=substrate_df,
    sequences_df=sequences_df,
    model=irrev_model,
    output_path=merged_data_output
)

display(merged_data)

### DEBUGGED UP TO THIS POINT - CHECK WHERE CPI-PRED KCAT VALUES ARE INCORPORATED 


# Step 3: Create kcat data file from CPI-Pred fold predictions
print("Obtaining all kcat values for reactions from CPI-Pred...")
processed_data_output = os.path.join(processed_data_dir, "ecoli_proccesed_data.csv")
fold_csv_paths = [
    os.path.join(CPIPred_data_dir, f"ecoli_predictions_fold_{i}.csv") 
    for i in range(1, 6)
]
processed_data = process_merged_data_with_folds(
    merged_data=merged_data,
    fold_csv_paths=fold_csv_paths,
    output_path=processed_data_output
)

# Step 4: Run optimization
print("Running optimization...")
# Set parameters for optimization
biomass_reaction = 'BIOMASS_Ec_iML1515_core_75p37M'  # Update with your biomass reaction ID
enzyme_upper_bound = 0.125  # gP/gDCW, enzyme fraction upper bound

# Run optimization
solution, flux_distribution = run_optimization_with_dataframe(
    model=irrev_model,
    processed_df=processed_data,
    objective_reaction=biomass_reaction,
    enzyme_upper_bound=enzyme_upper_bound,
    enzyme_ratio=True  # Using enzyme fraction constraint,
    output_dir=
)

print(f"Optimization complete. Biomass flux: {solution.objective_value}")

# Step 5: Plot results
print("Plotting flux distribution...")
# Plot flux distribution
fig = plot_flux_distribution(flux_distribution)
plt.show()

# Save results
flux_distribution.to_csv(os.path.join(processed_data_dir, "e_coli_flux_distribution.csv"), index=False)

print("Pipeline completed successfully!")

Preparing model data...
Loaded model with 95 reactions and 72 metabolites
Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp_72aqswc.lp


2025-03-28 12:50:37,950 - gurobipy - INFO - Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp_72aqswc.lp


Reading time = 0.01 seconds


2025-03-28 12:50:37,950 - gurobipy - INFO - Reading time = 0.01 seconds


: 72 rows, 190 columns, 720 nonzeros


...

,Single_gene,Name,Sequence
0,b1241,adhE,MAVTNVAELNALVERVKKAQREYASFTQEQVDKIFRAAALAAADAR...
1,b0351,mhpF,MSKRKVAIIGSGNIGTDLMIKILRHGQHLEMAVMVGIDPQSDGLAR...
2,s0001,s0001,None
3,b1849,purT,MTLLGTALRPAATRVMLLGSGELGKEVAIECQRLGVEVIAVDRYAD...
4,b3115,tdcD,MNEFPVVLVINCGSSSIKFSVLDASDCEVLMSGIADGINSENAFLS...
...,...,...,...
132,b0008,talB,MTDKLTSLRQYTTVVADTGDIAAMKLYQPQDATTNPSLILNAAQIP...
133,b2464,talA,MNELDGIKQFTTVVADSGDIESIRHYHPQDATTNPSLLLKAAGLSQ...
134,b2465,tktB,MSRKDLANAIRALSMDAVQKANSGHPGAPMGMADIAEVLWNDFLKH...
135,b2935,tktA,MSSRKELANAIRALSMDAVQKAKSGHPGAPMGMADIAEVLWRDFLK...


Merging substrate and sequence data...


,Reactions,Reaction_partners,SMILES,BiGG Name,DB Name,Cleaned Substrate,Single_gene,GPR_rules,SEQ
0,PFK,atp_c,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,ATP C10H12N5O13P3,ATP,atp,b3916,b3916 or b1723,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...
1,PFK,atp_c,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,ATP C10H12N5O13P3,ATP,atp,b1723,b3916 or b1723,MVRIYTLTLAPSLDSATITPQIYPEGKLRCTAPVFEPGGGGINVAR...
2,PFK,f6p_c,O=P([O-])([O-])OC[C@H]1O[C@](O)(CO)[C@@H](O)[C...,D-Fructose 6-phosphate,beta-D-Fructose 6-phosphate,f6p,b3916,b3916 or b1723,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...
3,PFK,f6p_c,O=P([O-])([O-])OC[C@H]1O[C@](O)(CO)[C@@H](O)[C...,D-Fructose 6-phosphate,beta-D-Fructose 6-phosphate,f6p,b1723,b3916 or b1723,MVRIYTLTLAPSLDSATITPQIYPEGKLRCTAPVFEPGGGGINVAR...
4,PFL,coa_c,CC(C)(CO[P](O)(=O)O[P](O)(=O)OC[C@H]1O[C@H]([C...,Coenzyme A,NaN,coa,b0902,((b0902 and b0903) and b2579) or (b0902 and b0...,MSVIGRIHSFESCGTVDGPGIRFITFFQGCLMRCLYCHNRDTWDTH...
...,...,...,...,...,...,...,...,...,...
598,MDH_reverse,nadh_c,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...,Nicotinamide adenine dinucleotide - reduced,NADH,nadh,b3236,b3236,MKVAVLGAAGGIGQALALLLKTQLPSGSELSLYDIAPVTPGVAVDL...
599,MDH_reverse,oaa_c,O=C([O-])CC(=O)C(=O)[O-],Oxaloacetate,oxaloacetate,oaa,b3236,b3236,MKVAVLGAAGGIGQALALLLKTQLPSGSELSLYDIAPVTPGVAVDL...
600,NH4t_reverse,nh4_c,[NH4+],Ammonium,NH4+,nh4,s0001,s0001 or b0451,None
601,NH4t_reverse,nh4_c,[NH4+],Ammonium,NH4+,nh4,b0451,s0001 or b0451,MKIATIKTGLASLAMLPGLVMAAPAVADKADNAFMMICTALVLFMT...


Running optimization...


KeyError: "Index 'BIOMASS_Ec_iML1515_core_75p37M' is not valid for indexed component 'reaction'"